# Specific Risk — Construction Spec (USE4-faithful)

**Purpose.** This notebook is the **source-of-truth spec** for building
`specific_risk_build.ipynb`. The specific-risk model forecasts the per-stock
idiosyncratic volatility σ_n that forms **Δ = diag(σ_n²)**, the second term of the
portfolio-risk identity

```
σ²_p = wᵀ X F Xᵀ w  +  wᵀ Δ w
       └── factor ──┘   └ specific ┘
```

With the factor covariance `F` built in `06_fcov`, Δ completes the model. The forecast
is the full USE4 stack over the **coverage universe**: time-series (EWMA + Newey–West on
daily specific returns) → structural model (for thin-history names) → coverage blend →
Bayesian shrinkage by size decile → volatility-regime multiplier. Validated by
specific-risk bias statistics, **by size decile**.

**Audience.** You — plus whatever AI assistant you are driving. Treat its output as
deeply untrustworthy until audited. Follow the stages linearly.

**Why Δ is diagonal.** The CSR's WLS first-order conditions force `Xᵀ W u = 0`: specific
returns are cap-weighted orthogonal to every factor over ESTU, so the systematic/specific
cross-covariance vanishes. Treating distinct stocks' specific returns as mutually
uncorrelated (the standard assumption) makes the specific block diagonal. The non-ESTU
residuals are out-of-sample (not orthogonal), so their diagonal is an approximation.

**Kernels.** Implement the estimator kernels — the EWMA variance and its Newey–West
long-run version, the empirical-Bayes shrink, the retransformation constant `E0`, and
the bias statistic — as pure functions in a module of your own, and validate them with a
known-answer test script (synthetic data with known truth) **before** wiring them into
the build notebook (§5 enumerates the tests).

**Companion docs.** Reference audit `specific_risk_audit.md`; Layer-0 spec
`05_csr/daily_csr_spec.ipynb`; downstream consumer `08_risk_decomp/risk_decomp_spec.ipynb`.

> **Your task.** The spec and the reference audit are provided;
> `specific_risk_build.ipynb` is yours to write. The canonical pipeline order runs this
> stage after `06_fcov`, but it consumes **nothing** from 06 — its inputs are section
> 05's residual files plus the exposure deliverables from 01–04, so it is buildable as
> soon as `daily_csr_build.ipynb` has run. F and Δ first meet in `08_risk_decomp`.

## §1. Deliverable and output schema

**File:** `data/out/specific_risk.parquet`, zstd, statistics=True. One row per
coverage-universe stock-date (post warm-up).

| Column | Type | Description |
|---|---|---|
| `permaticker` | Int64 | Sharadar permanent ticker ID |
| `signal_date` | Date | Forecast (month-end) date t |
| `in_estu` | Bool | ESTU (regression universe) vs non-ESTU coverage stock |
| `sigma_ts` | Float64 | Layer 1 time-series forecast (null if no daily history) |
| `gamma` | Float64 | Coverage coefficient ∈ [0,1] — trust in `sigma_ts` |
| `sigma_str` | Float64 | Layer 2 structural forecast (from exposures) |
| `sigma_blend` | Float64 | Layer 3 blend γ·σ_ts + (1−γ)·σ_str |
| `sigma_sh` | Float64 | Layer 4 shrunk toward size-decile mean |
| `sigma_spec` | Float64 | **The deliverable** — Layer 5 regime-scaled σ_n (monthly) |
| `mcap` | Float64 | Market cap on signal_date |
| `size_decile` | Int8 | Per-date mcap decile (0 small → 9 large) |

Δ_t = diag(`sigma_spec`²) over the coverage stocks at t. The intermediate columns are kept
so each layer is independently auditable. Specific returns are **monthly** here; the
time-series layer estimates from **daily** residuals and scales to the monthly horizon.

## §2. Pipeline

```
STAGE 0  Setup, parameters
STAGE 1  Inputs: daily ESTU residuals (csr_specific_returns_daily), coverage exposure
         matrix (industries_use4 + 12 styles, complete-case), monthly calendar
STAGE 2  Layer 1 — time-series σ_ts: daily EWMA + Newey–West, ×21 → monthly; snapshot
         to each month-end (last daily obs ≤ t); coverage coefficient γ
STAGE 3  Layer 2 — structural σ_str: per date, WLS log(σ_ts) ~ exposures over good-history
         names; predict for all coverage stocks; E0 retransformation correction
STAGE 4  Layer 3 — blend: σ_blend = γ·σ_ts + (1−γ)·σ_str  (drop pre-2001 warm-up)
STAGE 5  Layer 4 — shrinkage: empirical-Bayes toward size-decile means
STAGE 6  Layer 5 — vol-regime: daily causal multiplier λ; σ_spec = λ·σ_sh; ship
STAGE 7  Validate (positivity, coverage, bias stats overall + by decile)
```

**PIT discipline.** At each month-end t the forecast uses only data dated ≤ t: daily
residuals through t (strict backward as-of snapshot), the structural fit and shrinkage
groups formed at t, and the regime multiplier accumulated causally. The warm-up (before the
daily residuals begin, ~pre-2001) carries no forecast and is dropped — exactly as the
covariance forecast drops its own warm-up.

**Layer 0 (upstream).** The daily ESTU specific returns `u_{n,d}` are the regression
residuals emitted by `daily_csr_build.ipynb` (spec: `05_csr/daily_csr_spec.ipynb`) — the
daily-frequency analogue the covariance forecast already relies on.

**Position in the pipeline.** This stage reads the daily residuals **directly from
section 05**, not via `06_fcov`: 06 consumes the daily *factor* returns, this stage the
daily *residuals* — independent consumers of the same daily CSR. Inputs on disk:
`data/out/csr_specific_returns_daily.parquet` (`permaticker, date, u` — Layer 0);
`data/out/country_use4.parquet` (anchor: monthly calendar + `mcap`);
`data/out/industries_use4.parquet` (`in_estu`, `mcap > 0`) + the 12 `<style>_use4.parquet`
files joined on (`permaticker`, `signal_date`) — the complete-case coverage matrix; and
`data/out/csr_specific_returns.parquet` (monthly realized u — **validation only**, Stage 7).

## §3. STAGE 0 — Parameters

```python
# Layer 1 — time-series (daily, obs-sequence decay; ESTU names are liquid)
H_VOL     = 84      # EWMA half-life (trading days) for daily specific variance
NW_LAGS   = 5       # Newey–West Bartlett lags (one trading week)
HORIZON   = 21      # trading days/month (daily variance ×21 → monthly)
NW_FLOOR_FRAC, NW_CAP_FRAC = 0.25, 4.0   # bound V_NW ∈ [0.25, 4]·V0
VAR_FLOOR = 1e-12   # 💡 absolute variance backstop below the NW bounds
SIGMA_MIN = 0.01    # minimum monthly specific vol (~3.5%/yr)

# Coverage coefficient γ
TS_FULL        = 252   # daily obs for full TS trust (depth)
GAMMA_REC_DAYS, GAMMA_REC_FADE = 35, 60   # recency: full within 35d, fading to 0 by 95d

GOOD_GAMMA = 0.90      # γ threshold for the structural "good-history" fit set
N_DECILES, SHRINK_Q = 10, 1.0             # Layer 4 shrinkage
H_VRA, Z_CAP = 42, 10.0                    # Layer 5 regime EWMA half-life; winsorize
```

**Complete-case (estimate-free).** The model consumes the complete-case residuals and
exposures built upstream: the coverage matrix keeps only stock-dates with all 12 style
scores present and finite and `mcap > 0`. If you later choose to impute missing
exposures, revisit the coverage counts here — this lab's design stance is complete-case
throughout, and the structural model already serves the thin-history and non-ESTU names,
whose problem is residual history, not exposure gaps.

## §4. The five layers

### Layer 1 — time-series (`sigma_ts`)

✅ **PDF SPEC.** Specific risk is forecast from the history of specific returns with the same
recency-weighting + serial-correlation machinery as the factor covariance, per stock.

For each stock, the mean-zero EWMA variance `V0 = ewm(u², H_VOL)`; the Newey–West long-run
variance `V_NW = V0 + 2 Σ_l (1−l/(L+1)) γ_l` with `γ_l` the EWMA lag-l autocovariance;
`σ_ts = √(21 · V_NW)`. Snapshot to each month-end via a backward as-of join (last daily obs
≤ t); the EWMA at t equals the EWMA at that last obs (the decay cancels in the ratio).

❓ **NOT IN PDF — decay basis.** 💡 Obs-sequence half-life (dataframe-engine `ewm_mean`),
not calendar-trading-day age. ESTU names are liquid, so the two coincide.

❓ **NOT IN PDF — NW collapse.** Specific returns carry bid-ask bounce (negative
autocovariance) that can drive a scalar NW estimate to zero or negative. 💡 Bound
`V_NW ∈ [0.25, 4]·V0` (plus the absolute `VAR_FLOOR` backstop), and floor the monthly vol
at `SIGMA_MIN` (below that is a quiet-window artifact, not a riskless stock).

### Layer 2 — structural (`sigma_str`)

❓ **NOT IN PDF — thin histories.** Many names (IPOs, illiquid, non-ESTU) have too little
history for a reliable `σ_ts`, and they carry the *highest* true specific risk. 💡 Learn the
map from exposures to specific risk on the good-history names and apply it to everyone:

```
log σ_ts = xₙᵀ b + ε    (WLS, √mcap-weighted, over γ ≥ GOOD_GAMMA names)
σ_str    = E0 · exp(xₙᵀ b̂)    for every coverage stock
```

`xₙ` = intercept + the 12 style exposures. `E0` corrects the lognormal retransformation
bias by matching the cap-weighted structural level to the cap-weighted time-series level:
`E0 = Σ(w·σ_ts) / Σ(w·exp(xᵀb̂))` over the fit set, w = mcap. Fitting in logs tames the
right skew; forgetting `E0` biases every forecast low. 💡 Degradation ladder: if the
good-history set has < 50 names, fit on all names with a valid `σ_ts`; if still < 13
(intercept + 12 styles), `σ_str` is null for that date. Implementation note: a √mcap WLS
weight means scaling design **and** target rows by `mcap^(1/4)` before least squares —
the implied weight is the square of the row scale, an easy thing to mis-modify.

### Layer 3 — blend (`sigma_blend`)

`σ_blend = γ·σ_ts + (1−γ)·σ_str` where both parts exist and γ > 0; otherwise
`coalesce(σ_str, σ_ts)`. γ = depth × recency: `min(1, obs/TS_FULL)` times a recency
factor (full within `GAMMA_REC_DAYS`, fading to 0 over `GAMMA_REC_FADE`; γ null-filled
to 0). Full reliable history → γ=1 (pure TS); none → γ=0 (pure structural). A smooth γ
avoids a cliff at the history cutoff. Rows with a null blend (pre-2001 warm-up, before
the daily residuals begin) are dropped here.

### Layer 4 — shrinkage (`sigma_sh`)

Even good `σ_ts` over-disperses in the cross-section (regression to the mean). Shrink toward
the size-decile cap-weighted mean `μ`:

```
σ_sh = v·μ + (1−v)·σ_blend ,   v = q|σ−μ| / (Δ + q|σ−μ|)
```

with Δ the within-decile dispersion (unweighted std of `σ_blend`; v = 0 if Δ ≤ 0) and
q = `SHRINK_Q`. Names far from their peers (likely noise) are pulled hard; names near μ
barely move — empirical Bayes / James–Stein. Deciles are per-date ordinal mcap ranks
(0 small → 9 large).

### Layer 5 — volatility-regime (`sigma_spec`)

A daily, causal cross-sectional bias thermometer `B_t = Σ_n w_n (u_{n,t}/(σ_sh/√21))²`
(cap-weighted, winsorized at ±`Z_CAP`); `λ = √EWMA_{H_VRA}(B)`; `σ_spec = λ·σ_sh`. A uniform
scalar — it resets the level without disturbing the cross-sectional shape Layers 2–4 set, so
it is orthogonal to the shrinkage. 💡 PIT detail: day d is standardized by the `σ_sh` of the
last month-end **strictly before** d; λ is sampled at each month-end by backward as-of
(1.0 before any daily data exists), and the shipped `σ_spec` is floored at `SIGMA_MIN`
once more.

## §5. Validation contract

The bias statistic standardizes the realized monthly specific return `u(t→t+1)` by the
forecast `σ_spec(t)` (strictly out-of-sample; realized u and the `w_reg` cap weights come
from `data/out/csr_specific_returns.parquet`); a calibrated forecast gives a cap-weighted
bias ≈ 1. The standardized return is **winsorized at ±Z_CAP** (the regime-stat cap): a
cap-weighted RMS is otherwise non-robust to a single catastrophic idiosyncratic move at the
warm-up edge (a name with one month of daily history, so its TS vol is floored) — without it
one ~−80% distressed-stock collapse in March 2001 inflated the *time-series-only* decile-6
statistic to 1.37. The deliverable is never affected (that name's γ leans it on the structural
model).

| check | gate |
|---|---|
| 1 | `sigma_spec` strictly positive, finite, complete coverage (no nulls) |
| 2 | overall cap-weighted bias ∈ [0.85, 1.15] |
| 3 | bias profile flat across size deciles (spread < 0.40, each decile in [0.7, 1.4]) |
| 4 | **headline:** the full stack flattens the time-series-only bias slope — the decile spread and the small-cap (deciles 0–2) mean \|bias−1\| both fall vs TS-only |
| 5 | sane levels: ESTU **median** specific vol in a plausible annualized band [10%, 45%] |

Check 4 is the load-bearing result — the specific-risk analogue of the eigenfactor smile.
A time-series-only forecast under-predicts the small-cap deciles (bias 1.60, 1.51, 1.24 in
deciles 0–2); the structural model and shrinkage flatten the profile onto 1 ([1.02, 1.11],
spread 0.09), which is exactly where the time series fails. Render the bias-by-decile
figure (TS-only vs full stack) as part of your validation battery — it is the one picture
that shows the stack working.

### Kernel known-answer tests (write these first)

Validate your kernel module with a seeded known-answer test script before the build ever
imports it. The reference battery:

1. **EWMA variance recovers an iid variance** — N(0, 0.3²) draws with a near-flat
   half-life ≫ n: estimate within ±0.01 of 0.09.
2. **Newey–West lifts an AR(1) toward its long-run variance** — ρ = 0.5: the plain EWMA
   variance ≈ Var(u) = 1.333, and NW with generous lags rises above it toward 4.0
   (`V0 < V_NW` and `V_NW > 3.0`).
3. **Shrink** — identity at the group mean; monotone and bounded (a near-mean name barely
   moves, a far outlier is pulled hard, the result lies strictly between σ and μ); a
   larger q shrinks harder.
4. **E0** — with lognormal noise of scale τ, `E0 ≈ exp(τ²/2)`; applying it ties the
   cap-weighted structural level to the cap-weighted TS level (to 1e-9 relative).
5. **Bias statistic ≈ 1 under truth** — realized = forecast · N(0,1) gives 1 ± 0.02.

Note the production Layer-1 pass runs vectorized in your dataframe engine (per-stock
EWMA/NW over ~13M daily rows); the numpy kernels are known-answer *references* — their
lag-covariance normalizations need not be bit-identical, so validate levels, not bits.

## §6. Master list of ❓ NOT-IN-PDF decisions

USE4 fixes the *structure* (TS + structural + shrinkage + vol-regime); the parametrization
is the implementer's. The decisions and where to revisit:

| # | Decision | Default chosen | Where to revisit |
|---|---|---|---|
| 1 | Daily decay basis | Obs-sequence half-life (not calendar age) | If thin/gappy names enter the TS set |
| 2 | NW collapse on bounce | Bound V_NW ∈ [0.25, 4]·V0 | If a non-monthly horizon changes the bounce/signal balance |
| 3 | Minimum specific vol | Floor at 1%/mo (~3.5%/yr) | If a genuinely lower-risk universe appears |
| 4 | Coverage coefficient γ | depth `min(1,obs/252)` × recency (35d→95d fade) | If the blended profile shows a discontinuity in history length |
| 5 | Structural regressors | 12 style scores + intercept, √mcap-weighted | If industry adds explanatory power |
| 6 | Shrinkage groups / q | Size deciles; q = 1 | Calibrate q against the by-decile bias if the small-cap tail drifts |
| 7 | Vol-regime cadence | Daily bias stat, winsorized ±10, EWMA half-life 42d | If a monthly stat suffices, or crashes are missed |
| 8 | Bias-stat winsorization | Standardized return clipped at ±10 (same cap) | If a less/more robust validation metric is wanted |
| 9 | Warm-up | Drop month-ends before any structural fit (pre-2001) | Self-heals; the daily residuals define the start |

The overall bias sits slightly above 1 (~1.07, a small in-band under-forecast); the lever to
centre it is the EWMA half-life or a small global calibration, both trivially tunable now
that the bias battery is wired into the build's validation stage. The flat decile profile —
the hard part — is in hand.